# 05 — Grey Wolf Optimizer (GWO) Feature Selection

Binary GWO searches for a compact feature mask. The implementation keeps the best-so-far wolf instead of accidentally returning only the best wolf from the last iteration.


In [1]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "utils").exists():
    REPO_ROOT = REPO_ROOT.parent

sys.path.insert(0, str(REPO_ROOT))

import random
import numpy as np


## Binary Grey Wolf Optimizer

In [2]:
# -------------------------------------------------
# Binary Grey Wolf Optimizer (BGWO)
# -------------------------------------------------

import numpy as np


def sigmoid(x):
    return 1 / (1 + np.exp(-x))


def run_bgwo(
    obj_func,
    n_features,
    pop_size=30,
    iterations=50,
):

    # Initialize wolves
    positions = np.random.randint(0, 2, (pop_size, n_features))

    fitness = np.array([obj_func(w) for w in positions])

    order = np.argsort(fitness)

    alpha = positions[order[0]].copy()
    beta = positions[order[1]].copy()
    delta = positions[order[2]].copy()

    # Preserve the best solution found across every iteration.
    best_position = alpha.copy()
    best_score = float(fitness[order[0]])

    convergence = []

    for t in range(iterations):

        a = 2 - 2 * (t / iterations)

        for i in range(pop_size):

            new_position = np.zeros(n_features)

            for j in range(n_features):

                # -------- Alpha --------
                r1 = np.random.rand()
                r2 = np.random.rand()

                A1 = 2 * a * r1 - a
                C1 = 2 * r2

                D_alpha = abs(C1 * alpha[j] - positions[i, j])
                X1 = alpha[j] - A1 * D_alpha

                # -------- Beta --------
                r1 = np.random.rand()
                r2 = np.random.rand()

                A2 = 2 * a * r1 - a
                C2 = 2 * r2

                D_beta = abs(C2 * beta[j] - positions[i, j])
                X2 = beta[j] - A2 * D_beta

                # -------- Delta --------
                r1 = np.random.rand()
                r2 = np.random.rand()

                A3 = 2 * a * r1 - a
                C3 = 2 * r2

                D_delta = abs(C3 * delta[j] - positions[i, j])
                X3 = delta[j] - A3 * D_delta

                X = (X1 + X2 + X3) / 3

                prob = sigmoid(X)

                new_position[j] = 1 if np.random.rand() < prob else 0

            # Prevent empty feature subset
            if new_position.sum() == 0:
                new_position[np.random.randint(n_features)] = 1

            positions[i] = new_position

        # Evaluate
        fitness = np.array([obj_func(w) for w in positions])

        order = np.argsort(fitness)

        alpha = positions[order[0]].copy()
        beta = positions[order[1]].copy()
        delta = positions[order[2]].copy()

        current_best_score = float(fitness[order[0]])

        if current_best_score < best_score:
            best_score = current_best_score
            best_position = alpha.copy()

        convergence.append(best_score)

    return best_position, best_score, convergence

## Run the complete feature-selection experiment

The shared experiment runner supplies the configured datasets, classifiers,
optimizer seeds, population size, and iteration count. Feature selection uses
the validation set. The test set is evaluated only after the final mask has
been selected.


In [3]:
from utils.experiments import run_feature_selector


def gwo_runner(
    objective,
    n_features,
    pop_size,
    iterations,
):
    return run_bgwo(
        obj_func=objective,
        n_features=n_features,
        pop_size=pop_size,
        iterations=iterations,
    )


gwo_results = run_feature_selector("GWO", gwo_runner)
gwo_results.tail()


Saved: ('breast', 'svm', 0)


Saved: ('breast', 'random_forest', 0)


Saved: ('breast', 'xgboost', 0)


Saved: ('heart', 'svm', 0)


Saved: ('heart', 'random_forest', 0)


Saved: ('heart', 'xgboost', 0)


,Dataset,Classifier,Algorithm,Seed,ValidationFitness,Accuracy,Precision,Recall,F1,ROC_AUC,Features,SelectionRuntime,TestRuntime,SelectedFeatureNames,MaskFile,ConvergenceFile
1,breast,random_forest,GWO,0,0.031053,0.982456,1.000000,0.952381,0.975610,0.998347,15,5.457747,0.259268,"[""radius_mean"", ""perimeter_mean"", ""smoothness_...",results/smoke/artifacts/breast__random_forest_...,results/smoke/artifacts/breast__random_forest_...
2,breast,xgboost,GWO,0,0.031719,0.947368,0.909091,0.952381,0.930233,0.984127,17,2.363222,0.085223,"[""radius_mean"", ""texture_mean"", ""area_mean"", ""...",results/smoke/artifacts/breast__xgboost__gwo__...,results/smoke/artifacts/breast__xgboost__gwo__...
3,heart,svm,GWO,0,0.145891,0.815217,0.846939,0.813725,0.830000,0.898195,15,0.425846,0.021668,"[""trestbps"", ""thalch"", ""oldpeak"", ""ca"", ""sex_F...",results/smoke/artifacts/heart__svm__gwo__seed0...,results/smoke/artifacts/heart__svm__gwo__seed0...
4,heart,random_forest,GWO,0,0.146291,0.782609,0.829787,0.764706,0.795918,0.868604,16,6.657647,0.327716,"[""trestbps"", ""chol"", ""thalch"", ""oldpeak"", ""ca""...",results/smoke/artifacts/heart__random_forest__...,results/smoke/artifacts/heart__random_forest__...
5,heart,xgboost,GWO,0,0.140511,0.771739,0.833333,0.735294,0.781250,0.889825,15,1.999753,0.086269,"[""chol"", ""oldpeak"", ""ca"", ""sex_Female"", ""sex_M...",results/smoke/artifacts/heart__xgboost__gwo__s...,results/smoke/artifacts/heart__xgboost__gwo__s...
